# 08 - External Index Retrievers 🌐

## Learning Objectives 🎯

In this notebook, you'll learn:

1. **What are External Index Retrievers** and how they differ from vector store retrievers
2. **ArxivRetriever** - Search and retrieve scholarly articles from arxiv.org
3. **WikipediaRetriever** - Access Wikipedia articles for general knowledge
4. **TavilySearchAPIRetriever** - Perform real-time internet searches
5. **Integration with RAG Chains** - Combine external retrievers with LLMs
6. **Best Practices** - When and how to use each retriever effectively

---

## Table of Contents 📚

1. [Introduction to External Retrievers](#intro)
2. [Setup & Installation](#setup)
3. [ArxivRetriever - Academic Papers](#arxiv)
4. [WikipediaRetriever - General Knowledge](#wikipedia)
5. [TavilySearchAPIRetriever - Web Search](#tavily)
6. [Integration with RAG Chains](#rag)
7. [Comparison & Use Cases](#comparison)
8. [Best Practices](#best-practices)
9. [Summary & Exercises](#summary)

---

<a id='intro'></a>
## 1. Introduction to External Index Retrievers 🔍

### What are External Index Retrievers?

**External Index Retrievers** search over external data sources (e.g., the internet, academic databases, knowledge bases) rather than your local vector store.

### Key Differences:

| Feature | Vector Store Retrievers | External Index Retrievers |
|---------|------------------------|---------------------------|
| **Data Source** | Your embedded documents | External databases/APIs |
| **Data Freshness** | Static (at indexing time) | Real-time or regularly updated |
| **Setup Required** | Embedding + Vector store | API keys (sometimes) |
| **Use Cases** | Internal documents, knowledge bases | Current events, academic research, general knowledge |
| **Cost** | Embedding cost + storage | API calls (often free tier available) |

### When to Use External Retrievers:

- ✅ You need **up-to-date information** from the internet
- ✅ You want to access **specialized databases** (e.g., academic papers)
- ✅ You need **general knowledge** without building a custom knowledge base
- ✅ You want to **augment** your local data with external sources

---

<a id='setup'></a>
## 2. Setup & Installation ⚙️

### Required Packages

All external retrievers are part of `langchain-community`. You'll also need:

```bash
pip install langchain-community
pip install arxiv           # For ArxivRetriever
pip install wikipedia       # For WikipediaRetriever
pip install tavily-python   # For TavilySearchAPIRetriever
```

### Environment Variables

For TavilySearchAPIRetriever, you'll need an API key:

```
TAVILY_API_KEY=your_api_key_here
```

Get your free API key at: https://tavily.com/

---

In [40]:
# Setup: Import required libraries
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Import LangChain components
from langchain_community.retrievers import TavilySearchAPIRetriever
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_arxiv import ArxivRetriever
from langchain_perigon import WikipediaRetriever

# Verify versions
import langchain
print(f"✅ LangChain version: {langchain.__version__}")
print("✅ Setup complete!")

✅ LangChain version: 1.3.14
✅ Setup complete!


<a id='arxiv'></a>
## 3. ArxivRetriever - Academic Papers 📄

### 🔰 BEGINNER: What is ArxivRetriever?

**ArxivRetriever** searches [arxiv.org](https://arxiv.org), a repository of electronic preprints for research papers in:
- Physics
- Mathematics
- Computer Science
- Quantitative Biology
- Quantitative Finance
- Statistics

### Use Cases:
- 📚 Literature review for research
- 🧠 Getting latest research on AI/ML topics
- 📊 Finding papers by specific authors
- 🔬 Accessing cutting-edge research

---

### 🔰 BEGINNER: Basic ArxivRetriever Usage

In [20]:
!pip install arxiv
!python -m pip install langchain-arxiv-retriever


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# Create an ArxivRetriever instance
# By default, it returns top 3 documents
# Create retriever
arxiv_retriever = ArxivRetriever(
    k=3
)

# Search
query = "large language models"
docs = arxiv_retriever.invoke(query)

print(f"📚 Found {len(docs)} papers on '{query}'\n")

for i, doc in enumerate(docs, start=1):
    print("=" * 80)
    print(f"Paper {i}")
    print(f"Title: {doc.metadata.get('title', 'N/A')}")
    print(f"Authors: {doc.metadata.get('authors', 'N/A')}")
    print(f"Published: {doc.metadata.get('published', 'N/A')}")
    print(f"\nAbstract:")
    print(doc.page_content[:500])
    print("=" * 80)


📚 Found 3 papers on 'large language models'

Paper 1
Title: Enhancing Human-Like Responses in Large Language Models
Authors: ['Ethem Yağız Çalık', 'Talha Rüzgar Akkuş']
Published: 2025-01-09

Abstract:
This paper explores the advancements in making large language models (LLMs) more human-like. We focus on techniques that enhance natural language understanding, conversational coherence, and emotional intelligence in AI systems. The study evaluates various approaches, including fine-tuning with diverse datasets, incorporating psychological principles, and designing models that better mimic human reasoning patterns. Our findings demonstrate that these enhancements not only improve user interactio
Paper 2
Title: Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality
Authors: ['Yiming Ai', 'Zhiwei He', 'Ziyin Zhang', 'Wenhong Zhu', 'Hongkun Hao', 'Kai Yu', 'Lingjun Chen', 'Rui Wang']
Published: 2024-02-22

Abstract:
In this study, we delve into the 

### 🎓 INTERMEDIATE: Advanced ArxivRetriever Features

In [22]:
# Advanced: Retrieve more documents and explore metadata
arxiv_retriever_advanced = ArxivRetriever(
    k=5,  # Get top 5 papers
    # load_all_available_meta=True  # Load all metadata
)

# Search for papers on "transformers attention mechanism"
query = "transformers attention mechanism"
docs = arxiv_retriever_advanced.invoke(query)

print(f"📚 Retrieved {len(docs)} papers\n")

# Display metadata for all papers
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.metadata.get('Title', 'N/A')}")
    print(f"   Authors: {doc.metadata.get('Authors', 'N/A')}")
    print(f"   Published: {doc.metadata.get('Published', 'N/A')}")
    print(f"   Entry ID: {doc.metadata.get('entry_id', 'N/A')}")
    print()

📚 Retrieved 5 papers

1. N/A
   Authors: N/A
   Published: N/A
   Entry ID: http://arxiv.org/abs/2303.15105v1

2. N/A
   Authors: N/A
   Published: N/A
   Entry ID: http://arxiv.org/abs/2002.00741v1

3. N/A
   Authors: N/A
   Published: N/A
   Entry ID: http://arxiv.org/abs/2511.13780v1

4. N/A
   Authors: N/A
   Published: N/A
   Entry ID: http://arxiv.org/abs/2402.04161v2

5. N/A
   Authors: N/A
   Published: N/A
   Entry ID: http://arxiv.org/abs/2412.06439v1



### 🎓 INTERMEDIATE: Using .batch() for Multiple Queries

In [23]:
# Batch processing: Search multiple topics at once
queries = [
    "RAG retrieval augmented generation",
    "vector embeddings",
    "prompt engineering"
]

arxiv_retriever_batch = ArxivRetriever(k=3)
batch_results = arxiv_retriever_batch.batch(queries)

print("📚 Batch Search Results:\n")
for query, docs in zip(queries, batch_results):
    print(f"Query: '{query}'")
    print(f"  → Found {len(docs)} papers")
    if docs:
        print(f"  → Top result: {docs[0].metadata.get('Title', 'N/A')}")
    print()

📚 Batch Search Results:

Query: 'RAG retrieval augmented generation'
  → Found 3 papers
  → Top result: N/A

Query: 'vector embeddings'
  → Found 3 papers
  → Top result: N/A

Query: 'prompt engineering'
  → Found 3 papers
  → Top result: N/A



### 📊 Understanding ArxivRetriever Metadata

Each document returned by ArxivRetriever contains rich metadata:

```python
{
    'Published': '2023-06-15',           # Publication date
    'Title': 'Paper Title',              # Full title
    'Authors': 'Author1, Author2',       # Comma-separated authors
    'Summary': 'Abstract text...',       # Paper abstract/summary
    'entry_id': 'http://arxiv.org/...',  # Arxiv URL
}
```

The `page_content` field contains the full abstract/summary of the paper.

---

<a id='wikipedia'></a>
## 4. WikipediaRetriever - General Knowledge 📖

### 🔰 BEGINNER: What is WikipediaRetriever?

**WikipediaRetriever** searches and retrieves content from Wikipedia, the free encyclopedia with 6+ million articles.

### Use Cases:
- 🌍 General knowledge questions
- 📚 Quick facts and definitions
- 🏛️ Historical information
- 🧑‍🔬 Biographical data
- 🗺️ Geographic information

### Important Notes:
- ⚠️ Wikipedia content is **community-edited** - verify critical information
- ✅ Great for general knowledge, not for specialized or proprietary data
- 🌐 Supports multiple languages

---

### 🔰 BEGINNER: Basic WikipediaRetriever Usage

In [37]:
!pip install wikipedia
!pip install langchain-perigon


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain-0.3.30-py3-none-any.whl.metadata (6.4 kB)
  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.8 MB/s  0:00:00
Using cached packaging-25.0-py3-none-any.whl (66 kB)
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)

   ----------------------------------------  0/19 [py-cpuinfo]
   -- -------------------------------------  1/19 [wrapt]
   -- -------------------------------------  1/19 [wrapt]
   ---- -----------------------------------  2/19 [pluggy]
   ---- -----------------------------------  2/19 [pluggy]
  Attempting uninstall: packaging
   ---- -----------------------------------  2/19 [pluggy]
    Found existing installation: packaging 26.2
   ---- -----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-arxiv-retriever 1.0.0 requires langchain-core>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-community 0.4.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langchain-google-genai 4.3.2 requires langchain-core<2.0.0,>=1.5.1, but you have langchain-core 0.3.86 which is incompatible.
langchain-groq 1.1.3 requires langchain-core<2.0.0,>=1.4.0, but you have

In [41]:
# Create a WikipediaRetriever instance
# By default, it returns top 3 documents
wiki_retriever = WikipediaRetriever(k=2)

# Search for information on "Python programming language"
query = "Python programming language"
docs = wiki_retriever.invoke(query)

print(f"📖 Found {len(docs)} Wikipedia articles on '{query}'\n")

# Display first result
print("=" * 80)
print(f"Title: {docs[0].metadata.get('title', 'N/A')}")
print(f"Source: {docs[0].metadata.get('source', 'N/A')}")
print(f"\nContent (first 600 chars):\n{docs[0].page_content[:600]}...")
print("=" * 80)

📖 Found 2 Wikipedia articles on 'Python programming language'

Title: Python (programming language)
Source: wikipedia

Content (first 600 chars):
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with ...


### 🎓 INTERMEDIATE: Advanced WikipediaRetriever Features

In [42]:
# Advanced: Control number of results and document length
wiki_retriever_advanced = WikipediaRetriever(
    k=3,        # Get top 3 results
    # doc_content_chars_max=1000  # Limit content to 1000 characters per doc -----phase out
)

# Search for "Machine Learning"
query = "Machine Learning"
docs = wiki_retriever_advanced.invoke(query)

print(f"📖 Retrieved {len(docs)} Wikipedia articles\n")

# Display all results
for i, doc in enumerate(docs, 1):
    print(f"{i}. Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Summary: {doc.metadata.get('summary', 'N/A')[:150]}...")
    print(f"   Content length: {len(doc.page_content)} characters")
    print()

📖 Retrieved 3 Wikipedia articles

1. Title: Machine learning
   Summary: N/A...
   Content length: 921 characters

2. Title: Theoretical computer science
   Summary: N/A...
   Content length: 982 characters

3. Title: Predictive analytics
   Summary: N/A...
   Content length: 204 characters



### 🎓 INTERMEDIATE: Multilingual Support

In [43]:
# Search in different languages
# Default is English ('en'), but you can specify other languages

# Example: Search in Spanish
wiki_retriever_es = WikipediaRetriever(
    tk=1,
    lang="es"  # Spanish Wikipedia
)

query = "Inteligencia Artificial"
docs = wiki_retriever_es.invoke(query)

print(f"🌐 Search in Spanish Wikipedia: '{query}'\n")
print(f"Title: {docs[0].metadata.get('title', 'N/A')}")
print(f"Content preview:\n{docs[0].page_content[:400]}...")

🌐 Search in Spanish Wikipedia: 'Inteligencia Artificial'

Title: Artificial intelligence
Content preview:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use lear...


### 🎓 INTERMEDIATE: Batch Processing with WikipediaRetriever

In [44]:
# Batch search for multiple topics
queries = [
    "Albert Einstein",
    "Quantum Computing",
    "Neural Networks"
]

wiki_retriever_batch = WikipediaRetriever(k=1, doc_content_chars_max=500)
batch_results = wiki_retriever_batch.batch(queries)

print("📖 Batch Wikipedia Search Results:\n")
for query, docs in zip(queries, batch_results):
    print(f"Query: '{query}'")
    if docs:
        print(f"  → Title: {docs[0].metadata.get('title', 'N/A')}")
        print(f"  → Summary: {docs[0].page_content[:200]}...")
    print()

📖 Batch Wikipedia Search Results:

Query: 'Albert Einstein'
  → Title: Outline of Albert Einstein
  → Summary: The following outline is provided as an overview of and topical guide to Albert Einstein:
Albert Einstein – German-born theoretical physicist. He developed the theory of relativity, one of the two pil...

Query: 'Quantum Computing'
  → Title: Unconventional computing
  → Summary: Quantum computing, perhaps the most well-known and developed unconventional computing method, is a type of computation that utilizes the principles of quantum mechanics, such as superposition and enta...

Query: 'Neural Networks'
  → Title: Representation learning
  → Summary: Neural networks are a family of learning algorithms that use a "network" consisting of multiple layers of inter-connected nodes. It is inspired by the animal nervous system, where the nodes are viewed...



### 📊 Understanding WikipediaRetriever Metadata

Each document returned by WikipediaRetriever contains:

```python
{
    'title': 'Article Title',           # Wikipedia article title
    'summary': 'Brief summary...',       # Short summary (if available)
    'source': 'https://en.wikipedia...', # Full Wikipedia URL
}
```

The `page_content` field contains the article text (up to `doc_content_chars_max` characters).

---

<a id='tavily'></a>
## 5. TavilySearchAPIRetriever - Web Search 🔍

### 🔰 BEGINNER: What is TavilySearchAPIRetriever?

**TavilySearchAPIRetriever** performs **real-time internet searches** using the Tavily Search API, optimized for AI applications.

### Key Features:
- 🌐 **Real-time web search** - Get the latest information from the internet
- 🎯 **AI-optimized** - Returns clean, relevant content for LLMs
- 🔒 **Source attribution** - Includes URLs and metadata
- ⚡ **Fast & reliable** - Built specifically for AI use cases

### Use Cases:
- 📰 Current events and news
- 💹 Stock prices and market data
- 🌦️ Weather information
- 🏢 Company information
- 🔧 Technical documentation and tutorials

### Getting Started:
1. Sign up at https://tavily.com/ (free tier available)
2. Get your API key
3. Add to `.env` file: `TAVILY_API_KEY=your_api_key`

---

### 🔰 BEGINNER: Basic TavilySearchAPIRetriever Usage

In [ ]:
!uv pip install tavily-python

Using Python 3.12.10 environment at: D:\Learning\learning_production_ai\venv
Resolved 13 packages in 1.65s
Prepared 1 package in 134ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 63ms
 + tavily-python==0.7.27


In [33]:
# Create a TavilySearchAPIRetriever instance
# Make sure TAVILY_API_KEY is set in your .env file

tavily_retriever = TavilySearchAPIRetriever(k=3)  # Return top 3 results

# Search for "latest developments in artificial intelligence 2024"
query = "latest developments in artificial intelligence 2024"
docs = tavily_retriever.invoke(query)

print(f"🔍 Found {len(docs)} web results for '{query}'\n")

# Display first result
print("=" * 80)
print(f"Source: {docs[0].metadata.get('source', 'N/A')}")
print(f"\nContent (first 500 chars):\n{docs[0].page_content[:500]}...")
print("=" * 80)

🔍 Found 3 web results for 'latest developments in artificial intelligence 2024'

Source: https://www.trendmicro.com/en_us/research/25/a/top-ai-trends-from-2024-review.html

Content (first 500 chars):
‘Smallifying’ AI modelsHand in hand with the shift to agentic AI is the need for smaller, nimbler, faster models purpose-built for specific tasks. Again, lots of work went into this in 2024. In October, Meta released updates to its Llama AI model that are as much as four times faster and 56% smaller than their precursors, enabling sophisticated AI features on devices as small as smartphones. And Nvidia released its Nemotron-Mini-4B Instruct small language model (SLM), which gets VRAM usage down ...


### 🎓 INTERMEDIATE: Advanced TavilySearchAPIRetriever Features

In [34]:
# Advanced: Control search depth and domain filtering
from langchain_community.retrievers import TavilySearchAPIRetriever

# Advanced configuration
tavily_retriever_advanced = TavilySearchAPIRetriever(
    k=5,  # Return top 5 results
    # search_depth="advanced",  # "basic" or "advanced" (more thorough)
    # include_domains=["github.com", "stackoverflow.com"],  # Filter to specific domains
    # exclude_domains=["example.com"]  # Exclude specific domains
)

# Search for "LangChain tutorials"
query = "LangChain tutorials"
docs = tavily_retriever_advanced.invoke(query)

print(f"🔍 Retrieved {len(docs)} web results\n")

# Display all results with sources
for i, doc in enumerate(docs, 1):
    print(f"{i}. Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Content preview: {doc.page_content[:200]}...")
    print()

🔍 Retrieved 5 web results

1. Source: https://github.com/gkamradt/langchain-tutorials
   Content preview: 1. LangChain CookBook Part 1: 7 Core Concepts - Code, Video
2. LangChain CookBook Part 2: 9 Use Cases - Code, Video
3. Explore the projects below and jump into the deep dives

Prompt Engineering (my f...

2. Source: https://www.geeksforgeeks.org/data-science/langchain-tutorial
   Content preview: geeksforgeeks

 Interview Prep

 Data Science Tutorial
 Maths
 Statistics
 Big Data
 Machine Learning
 AI
 NumPy
 Pandas
 Data Analysis
 Deep Learning
 Data Mining
 Computer Vision

# LangChain Tutori...

3. Source: https://langchain-tutorials.com
   Content preview: LangChain Tutorials

🎓Free LangChain Tutorials - Learn AI Development Step-by-Step

# LangChain Tutorials

Learn LangChain with hands-on tutorials. Build RAG systems, work with vector databases, and c...

4. Source: https://www.youtube.com/watch?v=AOQyRiwydyo
   Content preview: # Langchain Tutorial For Beginners (2026 Guide) 

### 🎓 INTERMEDIATE: Real-Time Information Retrieval

In [35]:
# Example: Get current information (news, weather, stock prices, etc.)
from datetime import datetime

current_date = datetime.now().strftime("%B %d, %Y")

# Real-time queries
queries = [
    f"latest AI news {current_date}",
    "current weather in San Francisco",
    "NVIDIA stock price today"
]

tavily_realtime = TavilySearchAPIRetriever(k=2)

print(f"🕐 Real-Time Information (as of {current_date}):\n")

for query in queries:
    docs = tavily_realtime.invoke(query)
    print(f"Query: '{query}'")
    if docs:
        print(f"  → {docs[0].page_content[:250]}...")
        print(f"  → Source: {docs[0].metadata.get('source', 'N/A')}")
    print()

🕐 Real-Time Information (as of August 10, 2026):

Query: 'latest AI news August 10, 2026'
  → to

20 October 2026 4:00 pm

RAI

in

Amsterdam

## Related News

Developer Tech News

August 10, 2026

## Study finds LLM-native IDE security risks in system controls

AI News

August 10, 2026

## The limits of physics AI: where Siemens says the hum...
  → Source: https://www.artificialintelligence-news.com

Query: 'current weather in San Francisco'
  → {'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1786369357, 'localtime': '2026-08-10 06:42'}, 'current': {'last_...
  → Source: https://www.weatherapi.com/

Query: 'NVIDIA stock price today'
  → ## Stock Snapshot

NVIDIA(NVDA) stock is priced at $202.55, giving the company a market capitalization of 4.98T. It carries a P/E multiple of 31.76 and pays a dividend yield of 0.14%.

As of 2026-07-18, NVIDIA(NVDA) 

### 📊 Understanding TavilySearchAPIRetriever Metadata

Each document returned by TavilySearchAPIRetriever contains:

```python
{
    'source': 'https://example.com/...',  # Source URL
    'score': 0.95,                         # Relevance score (0-1)
    'title': 'Page Title',                 # Web page title (if available)
}
```

The `page_content` field contains the extracted text content from the web page.

---

<a id='rag'></a>
## 6. Integration with RAG Chains 🔗

Now let's combine external retrievers with LLMs to build powerful **Retrieval-Augmented Generation (RAG)** systems!

### 🔰 BEGINNER: Simple QA Chain with External Retriever

In [ ]:
# Build a simple RAG chain using WikipediaRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize components
wiki_retriever = WikipediaRetriever(top_k_results=2, doc_content_chars_max=2000)
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# Create prompt template
template = """Answer the question based on the following context from Wikipedia:

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL
rag_chain = (
    {"context": wiki_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ask a question
question = "What is quantum computing and how does it work?"
answer = rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}")

Question: What is quantum computing and how does it work?

Answer: - A quantum computer is a computer that uses quantum states—specifically superposition and entanglement—and the probabilistic outcomes of quantum measurements as resources for computing.
- The basic unit is the qubit, which can be in a superposition of 0 and 1, not just a single classical value.
- Computation proceeds by manipulating qubits with quantum operations to create interference among many possible states. This interference amplifies the probability of the desired measurement result.
- When you measure the qubits, you get one of the basis states with certain probabilities, dictated by the quantum state.
- The goal of quantum algorithms is to design procedures that steer the system so the correct outcomes become more likely (amplitude amplification), effectively exploring a huge number of possibilities in parallel.
- Quantum computers are not yet practical for everyday tasks; current hardware is largely experimen

### 🎓 INTERMEDIATE: Multi-Source RAG Chain

In [ ]:
# Advanced: Combine multiple retrievers for comprehensive answers
from langchain_core.runnables import RunnableParallel

# Initialize multiple retrievers
arxiv_retriever = ArxivRetriever(load_max_docs=2)
wiki_retriever = WikipediaRetriever(top_k_results=2, doc_content_chars_max=1500)

# Function to combine results from multiple retrievers
def multi_retriever(query):
    """Retrieve from multiple sources and combine results."""
    arxiv_docs = arxiv_retriever.invoke(query)
    wiki_docs = wiki_retriever.invoke(query)
    
    # Combine and format
    all_docs = []
    
    if arxiv_docs:
        all_docs.append("=== Academic Papers (ArXiv) ===")
        all_docs.extend([doc.page_content[:500] for doc in arxiv_docs])
    
    if wiki_docs:
        all_docs.append("\n=== General Knowledge (Wikipedia) ===")
        all_docs.extend([doc.page_content[:500] for doc in wiki_docs])
    
    return "\n\n".join(all_docs)

# Create multi-source RAG chain
multi_source_template = """Answer the question using information from multiple sources below:

{context}

Question: {question}

Provide a comprehensive answer that synthesizes information from both academic and general sources:"""

multi_prompt = ChatPromptTemplate.from_template(multi_source_template)

multi_rag_chain = (
    {"context": multi_retriever, "question": RunnablePassthrough()}
    | multi_prompt
    | llm
    | StrOutputParser()
)

# Ask a question
question = "What are transformers in machine learning?"
answer = multi_rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer (from multiple sources):\n{answer}")

Question: What are transformers in machine learning?

Answer (from multiple sources):
Transformers are a class of neural network architectures designed to process sequential data (most famously text) by using attention mechanisms to model relationships between elements in a sequence, without relying on traditional recurrence.

What makes transformers special (core ideas)
- Attention over tokens: The key idea is the attention mechanism, which lets the model weigh other positions in the input when encoding a given element (token). This allows the model to focus on the most relevant parts of the sequence for each token.
- Multi-head attention: Instead of a single attention view, transformers compute several attention patterns in parallel (heads). This lets the model capture different kinds of relationships and dependencies simultaneously.
- Embeddings and tokens: Text (or other sequential data) is first tokenized and each token is mapped to a vector via an embedding table. This provides a

### 🎓 INTERMEDIATE: Real-Time RAG with TavilySearchAPIRetriever

In [ ]:
# Build a RAG chain that uses real-time web search
tavily_retriever = TavilySearchAPIRetriever(k=3)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create prompt for real-time information
realtime_template = """Based on the latest information from the web:

{context}

Question: {question}

Provide an up-to-date answer having atleast 500 words with source attribution:"""

realtime_prompt = ChatPromptTemplate.from_template(realtime_template)

# Build real-time RAG chain
realtime_rag_chain = (
    {"context": tavily_retriever | format_docs, "question": RunnablePassthrough()}
    | realtime_prompt
    | llm
    | StrOutputParser()
)

# Ask a current events question
question = "What are the latest developments in AI regulation?"
answer = realtime_rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer (from real-time web search):\n{answer}")

Question: What are the latest developments in AI regulation?

Answer (from real-time web search):
As of the latest developments in AI regulation, countries around the world are increasingly recognizing the need for comprehensive laws and guidelines to govern the use of artificial intelligence. The European Union's AI Act, which was proposed in April 2021 and is set to come into effect on August 1, 2024, is one of the most significant pieces of AI regulation to date. This regulation aims to establish harmonized rules for AI across all 27 EU member states, addressing issues such as transparency, accountability, and data protection.

One of the key provisions of the EU AI Act is the creation of a European Artificial Intelligence Board, which will be responsible for overseeing the implementation of the regulation and ensuring compliance with its provisions. The board will also be tasked with issuing guidance on the application of the regulation and promoting cooperation between EU member s

---

<a id='comparison'></a>
## 7. Comparison & Use Cases 📊

### Retriever Comparison Table

| Feature | ArxivRetriever | WikipediaRetriever | TavilySearchAPIRetriever |
|---------|----------------|-------------------|-------------------------|
| **Data Source** | Academic papers (arxiv.org) | Wikipedia articles | Real-time web search |
| **API Key Required** | ❌ No | ❌ No | ✅ Yes (free tier) |
| **Data Freshness** | Recent research | Regularly updated | Real-time |
| **Best For** | Academic research, ML papers | General knowledge, definitions | Current events, news |
| **Content Type** | Research papers, abstracts | Encyclopedia articles | Web pages, news |
| **Default Results** | 3 papers | 3 articles | 5 results |
| **Multilingual** | ❌ No | ✅ Yes (300+ languages) | ✅ Yes |
| **Metadata** | Title, Authors, Published date | Title, Summary, URL | Source URL, Score |
| **Rate Limits** | Moderate | Moderate | API-dependent |
| **Cost** | 🆓 Free | 🆓 Free | 🆓 Free tier + paid |

---

### When to Use Each Retriever

#### ✅ Use **ArxivRetriever** when:
- You need peer-reviewed academic research
- You're building an AI/ML research assistant
- You want the latest scientific papers
- You need citations and author information

#### ✅ Use **WikipediaRetriever** when:
- You need general knowledge and definitions
- You want historical or biographical information
- You're building an educational chatbot
- You need multilingual support
- You want reliable, community-edited content

#### ✅ Use **TavilySearchAPIRetriever** when:
- You need real-time, up-to-date information
- You're answering current events questions
- You want to search the broader internet
- You need to filter by specific domains
- Your use case requires the latest data

---

### Combining Retrievers (Hybrid Approach)

For the most comprehensive RAG system:

```python
# Pseudo-code for hybrid retrieval
if query_type == "academic":
    use ArxivRetriever
elif query_type == "general_knowledge":
    use WikipediaRetriever
elif query_type == "current_events":
    use TavilySearchAPIRetriever
else:
    # Use multiple retrievers and combine results
    combine(ArxivRetriever, WikipediaRetriever, TavilySearchAPIRetriever)
```

---

<a id='best-practices'></a>
## 8. Best Practices 💡

### General Best Practices

#### 1. **Handle Errors Gracefully**

```python
try:
    docs = retriever.invoke(query)
except Exception as e:
    print(f"Error retrieving documents: {e}")
    docs = []  # Fallback to empty list
```

#### 2. **Set Appropriate Limits**

```python
# Don't retrieve too many documents (costs, latency)
arxiv_retriever = ArxivRetriever(load_max_docs=3)  # ✅ Good
arxiv_retriever = ArxivRetriever(load_max_docs=100)  # ❌ Too many
```

#### 3. **Cache Results for Repeated Queries**

```python
# Use a simple cache to avoid redundant API calls
from functools import lru_cache

@lru_cache(maxsize=100)
def cached_search(query: str):
    return retriever.invoke(query)
```

#### 4. **Verify Source Attribution**

```python
# Always include sources in your responses
for doc in docs:
    print(f"Source: {doc.metadata.get('source', 'N/A')}")
```

#### 5. **Combine with Vector Store Retrievers**

```python
# Use external retrievers for general knowledge
# Use vector stores for your proprietary data
def hybrid_retrieve(query):
    external_docs = wiki_retriever.invoke(query)
    internal_docs = vector_store.similarity_search(query)
    return external_docs + internal_docs
```

---

### Retriever-Specific Best Practices

#### ArxivRetriever:
- ✅ Use specific search terms (e.g., "BERT transformers" vs "AI")
- ✅ Limit results to 3-5 papers for LLM context
- ✅ Extract metadata for citations
- ❌ Don't use for non-academic queries

#### WikipediaRetriever:
- ✅ Use for general knowledge, not specialized topics
- ✅ Set `doc_content_chars_max` to avoid huge documents
- ✅ Verify information for critical use cases
- ❌ Don't rely on Wikipedia for real-time information

#### TavilySearchAPIRetriever:
- ✅ Monitor API usage (rate limits, costs)
- ✅ Use for time-sensitive queries
- ✅ Filter by domain for specific sources
- ❌ Don't use for queries that don't need real-time data

---

### Performance Tips

1. **Use `.batch()` for multiple queries**
   ```python
   # ✅ Efficient
   results = retriever.batch([q1, q2, q3])
   
   # ❌ Inefficient
   results = [retriever.invoke(q) for q in [q1, q2, q3]]
   ```

2. **Limit document length for LLM context**
   ```python
   # Truncate long documents to fit LLM context window
   docs = [Document(page_content=doc.page_content[:2000], metadata=doc.metadata) 
           for doc in raw_docs]
   ```

3. **Use async methods for concurrent retrieval** (if supported)
   ```python
   # For async-compatible retrievers
   import asyncio
   docs = await retriever.ainvoke(query)
   ```

---

<a id='summary'></a>
## 9. Summary & Exercises 📝

### 🎯 What You Learned

In this notebook, you learned:

✅ **External Index Retrievers** - Search over external data sources (internet, databases)

✅ **ArxivRetriever** - Retrieve academic papers from arxiv.org
   - Use cases: Research, ML papers, citations
   - Methods: `.invoke()`, `.batch()`
   - Metadata: Title, Authors, Published date

✅ **WikipediaRetriever** - Access Wikipedia articles
   - Use cases: General knowledge, definitions, history
   - Features: Multilingual support, customizable length
   - Metadata: Title, Summary, Source URL

✅ **TavilySearchAPIRetriever** - Real-time web search
   - Use cases: Current events, news, real-time data
   - Features: Domain filtering, search depth control
   - Metadata: Source URL, Relevance score

✅ **RAG Integration** - Combined external retrievers with LLMs
   - Built simple QA chains
   - Created multi-source RAG systems
   - Implemented real-time information retrieval

✅ **Best Practices** - Error handling, caching, source attribution

---

### 💪 Practice Exercises

#### Exercise 1: Academic Research Assistant (🔰 Beginner)
Create a RAG chain that:
- Uses `ArxivRetriever` to find papers on "deep learning"
- Extracts the top 3 paper titles and authors
- Summarizes each paper's abstract using an LLM

#### Exercise 2: Wikipedia Fact Checker (🔰 Beginner)
Build a system that:
- Takes a statement as input (e.g., "Python was created in 1991")
- Uses `WikipediaRetriever` to search for relevant articles
- Uses an LLM to verify if the statement is accurate

#### Exercise 3: Multi-Source News Aggregator (🎓 Intermediate)
Create a RAG chain that:
- Uses `TavilySearchAPIRetriever` to get latest AI news
- Uses `WikipediaRetriever` to get background on AI topics
- Combines both sources to provide a comprehensive news summary

#### Exercise 4: Hybrid Retrieval System (🎓 Intermediate)
Build a system that:
- Classifies queries into "academic", "general", or "current_events"
- Routes to the appropriate retriever based on query type
- Returns results from the most relevant source

#### Exercise 5: Multilingual Knowledge Base (🚀 Advanced)
Create a system that:
- Detects the language of the user's query
- Uses `WikipediaRetriever` with the appropriate language setting
- Returns answers in the user's language

---

### 🔗 Next Steps

- **Notebook 09**: Advanced Retrieval Techniques (Hybrid Search, Re-ranking)
- **Notebook 10**: Production RAG Systems (Caching, Monitoring, Scaling)
- **LangChain Documentation**: https://python.langchain.com/docs/integrations/retrievers/

---

### 📚 Additional Resources

- **ArXiv**: https://arxiv.org/
- **Wikipedia API**: https://www.mediawiki.org/wiki/API:Main_page
- **Tavily API**: https://tavily.com/
- **LangChain Retrievers**: https://python.langchain.com/docs/modules/data_connection/retrievers/

---

**Congratulations!** 🎉 You've mastered external index retrievers in LangChain!

You can now build RAG systems that access:
- 📄 Academic research (ArXiv)
- 📖 General knowledge (Wikipedia)
- 🌐 Real-time web data (Tavily)

Keep experimenting and building amazing AI applications! 🚀